# Partie 4 — GraphFrames

Projet Écosystème Big Data 2 — FastFood.ma — EMSI 5ème année, 2026-2027.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = Path("/work")
sys.path[:0] = [str(root / "src"), str(root / "third_party")]


Sommets : clients (`C-…`) et restaurants (`R-…`). Une arête client → restaurant agrège le nombre de commandes livrées et le montant.
On ne garde que les `DELIVERED`, après `dropDuplicates(order_id, status)`.

`inDegree` d'un restaurant = nombre de clients distincts, parce qu'il n'y a qu'une arête par couple.
`outDegree` d'un client = nombre de restaurants différents.


In [ ]:
from fastfood.partie4 import run
from fastfood.session import build_spark

spark = build_spark("fastfood-partie4-graph", graph=True)
result = run(spark)
result["top_pagerank"], result["components"], result["recommendations"]


## PageRank n'est pas le top 5

Le top 5 de la partie 2 compte des commandes : R-157, R-188, R-013, R-147, R-012. PageRank (reset 0,15, 10 itérations) propage un poids le long des arêtes. Les trois premiers restent les mêmes. Ensuite le classement s'écarte : R-153 (245 livraisons, 211 clients) passe devant R-147 (249 livraisons, 215 clients), et R-158 (236 livraisons, 187 clients) passe devant R-012 (248 livraisons, 204 clients). Un client qui fréquente peu de restaurants envoie plus de masse à chacun.

Le saut aléatoire est global. Les quatre villes sont dans le même graphe, reliées seulement par ce saut, pas par des arêtes. On compare les rangs à l'intérieur de ce biais, on ne prétend pas que Casablanca et Tanger ont la même masse.

## Composantes, motifs, Pregel

Les clients ne commandent que dans leur ville : quatre grandes composantes, une par ville (la plus grande, Casablanca, a 3 178 sommets). Huit petites composantes sont un restaurant absent du référentiel et le ou les clients qui n'ont commandé que là. Les 200 restaurants du fichier ont tous au moins une livraison.

Le motif `(c)-[e1]->(r1); (c)-[e2]->(r2)` avec la même catégorie trouve les clients déjà fidèles à une catégorie. La recommandation propose à un client un restaurant de cette catégorie qu'il n'a pas encore essayé, classé par le nombre de clients qui font les deux.

`aggregateMessages` envoie le montant de l'arête vers le restaurant et le somme. Le `groupBy` sur `dst` doit donner le même chiffre, à l'arrondi près. C'est le modèle Pregel : superstep, message, agrégat, nouvel état.

GraphX est l'API RDD Scala (`Graph[VD, ED]`). Il n'y a pas de GraphX utilisable en PySpark. GraphFrames pose les sommets et les arêtes dans des DataFrames, donc les motifs passent par Catalyst.
